In [30]:
import numpy as np
from ucimlrepo import fetch_ucirepo

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

In [31]:
def load_data(id: int):
    dataset = fetch_ucirepo(id = id)

    name = dataset.metadata.name
    X = dataset.data.features
    y = dataset.data.targets

    return name, X, y

In [32]:
def eda(X: pd.DataFrame, y: pd.DataFrame):
    print(f"Feature set shape : {X.shape}")
    print(f"Target set shape : {y.shape}")
    print("=" * 50)

    print("\nNull Values")
    print(X.isna().sum())
    print(y.isna().sum())

    print("=" * 50)

    print("\nColumns")
    print(X.columns.tolist())

    X.columns = (
        X.columns
        .str.replace(r"^X\d+\s*", '', regex=True)
        .str.replace(' ', '-')

    )

    print("\nColumns after removing prefix")
    print(X.columns.tolist())

    print("=" * 50)

    print("\nInfo")
    print(X.info())
    print(y.info())

    



In [36]:
class LinearRegression:
    def __init__(self, learning_rate = 0.01, epochs = 1000):
        self.W = None
        self.b = 0.0
        self.lr = learning_rate
        self.cost_history = None
        self.epochs = epochs

    def predict(self, X):
        return X @ self.W + self.b

    def fit(self, X, y):
        X = np.asarray(X, dtype=float)
        y = np.asarray(y, dtype=float).reshape(-1, 1) 
        
        n_samples, n_features = X.shape

        self.W = np.zeros((n_features, 1))
        self.b = 0.0
        self.cost_history = []
        
        for epoch in range(self.epochs):
            prediction = X @ self.W + self.b

            error = prediction - y

            dw = (2 / n_samples) * (X.T @ error)
            db = (2 / n_samples) * np.sum(error)

            self.W -= self.lr * dw
            self.b -= self.lr * db

            if epoch % 100 == 0:
                prediction = X @ self.W + self.b
                cost = np.mean((prediction - y) ** 2)
                print(f"Epoch : {epoch}, cost = {cost}")
                self.cost_history.append(cost)

        return self

In [34]:
def evaluate(y_act, y_pred):
    y_act = np.asarray(y_act)
    y_pred = np.asarray(y_pred)

    mse = np.mean((y_act - y_pred) ** 2)
    mae = np.mean(np.abs(y_act - y_pred))
    r2 = 1 - ((np.sum((y_act - y_pred) ** 2)) / np.sum((y_act - np.mean(y_act)) ** 2))

    return {
        "Mean Squared Error": mse,
        "Mean Absolute Error": mae,
        "R2 Score": r2
    }

In [37]:
name, X, y = load_data(id=477)
print(f"Exploratory data analysis for {name} dataset")
eda(X, y)

print("=" * 50)
print("Linear Regression from scratch")
print("=" * 50)

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size= 0.2,
    random_state=42
)

print(f"X Training set shape: {X_train.shape}")
print(f"X Testing set shape: {X_test.shape}")

print(f"y Training set shape: {y_train.shape}")
print(f"y Testing set shape: {y_test.shape}")

scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

model = LinearRegression(learning_rate = 0.01, epochs = 1000)
model.fit(X_train_scaled, y_train)

y_pred = model.predict(X_test_scaled)

print(f"Weights: {model.W}")
print(f"Bias: {model.b}")

print("=" * 50)
print("Evaluation")
print("=" * 50)

for key, value in evaluate(y_test, y_pred).items():
    print(f"{key}: {value:.4f}")


Exploratory data analysis for Real Estate Valuation dataset
Feature set shape : (414, 6)
Target set shape : (414, 1)

Null Values
X1 transaction date                       0
X2 house age                              0
X3 distance to the nearest MRT station    0
X4 number of convenience stores           0
X5 latitude                               0
X6 longitude                              0
dtype: int64
Y house price of unit area    0
dtype: int64

Columns
['X1 transaction date', 'X2 house age', 'X3 distance to the nearest MRT station', 'X4 number of convenience stores', 'X5 latitude', 'X6 longitude']

Columns after removing prefix
['transaction-date', 'house-age', 'distance-to-the-nearest-MRT-station', 'number-of-convenience-stores', 'latitude', 'longitude']

Info
<class 'pandas.DataFrame'>
RangeIndex: 414 entries, 0 to 413
Data columns (total 6 columns):
 #   Column                               Non-Null Count  Dtype  
---  ------                               --------------  -----  